In [ ]:
import requests
from bs4 import BeautifulSoup

headers = {'User-Agent': 'Mozilla/5.0'}
url = "https://www.gundam-gcg.com/en/cards/index.php"
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')

print(f"Status: {response.status_code}")

# Find all 'Included In' (Set/Package) options
package_buttons = soup.find_all('a', class_='js-selectBtn-package')
sets = {}
for btn in package_buttons:
    val = btn.get('data-val')
    label = btn.text.strip()
    if val and val not in sets:
        sets[val] = label

print(f"Found {len(sets)} sets:")
for val, label in sets.items():
    print(f" - {val}: {label}")


In [ ]:
import json
import time
import urllib.parse
import requests
from bs4 import BeautifulSoup

def get_card_details(card_id):
    url = "https://www.gundam-gcg.com/en/cards/detail.php"
    headers = {'User-Agent': 'Mozilla/5.0'}
    params = {'detailSearch': card_id}
    
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, 'html.parser')
    
    data = {
        "id": card_id,
        "code": card_id,
        "rarity": "",
        "name": "",
        "images": { "small": "", "large": "" },
        "level": "",
        "cost": "",
        "color": "",
        "cardType": "",
        "effect": "",
        "zone": "",
        "trait": "",
        "link": "",
        "ap": "",
        "hp": "",
        "sourceTitle": "",
        "getIt": "",
        "set": {}
    }
    
    # 1. Rarity
    rarity_div = soup.find('div', class_='rarity')
    if rarity_div:
        # Remove any inner whitespace (e.g. "LR                +" -> "LR+")
        data['rarity'] = "".join(rarity_div.text.split())
        
    # 2. Name & Image
    img_el = soup.find('div', class_='cardImage')
    if img_el and img_el.find('img'):
        img = img_el.find('img')
        data['name'] = img.get('alt', '').strip()
        
        img_src = img.get('src', '').replace('../', '')
        img_url = f"www.gundam-gcg.com/en/{img_src}"
        val = f"https://images.weserv.nl/?url={img_url}"
        data['images']['small'] = val
        data['images']['large'] = val

    # 3. Stats and Info
    data_col = soup.find('div', class_='cardDataCol')
    if data_col:
        dls = data_col.find_all('dl', class_='dataBox')
        for dl in dls:
            tit = dl.find('dt', class_='dataTit')
            txt = dl.find('dd', class_='dataTxt')
            if tit and txt:
                key = tit.text.strip().lower()
                val = txt.text.strip()
                
                if 'lv.' in key: data['level'] = val
                elif 'cost' in key: data['cost'] = val
                elif 'color' in key: data['color'] = val
                elif 'type' in key: data['cardType'] = val
                elif 'zone' in key: data['zone'] = val.replace('\n', ' / ').replace('  ', ' ')
                elif 'trait' in key: data['trait'] = val
                elif 'link' in key: data['link'] = val
                elif 'ap' in key: data['ap'] = val
                elif 'hp' in key: data['hp'] = val
                elif 'source title' in key: data['sourceTitle'] = val
                elif 'where to get it' in key: 
                    data['getIt'] = val
                    parts = val.split('[')
                    if len(parts) == 2:
                        data['set'] = {
                            "id": parts[1].replace(']', '').lower(),
                            "name": parts[0].strip()
                        }
                    else:
                        data['set'] = {"id": val.lower().replace(' ', '_'), "name": val}

        # 4. Effect Text
        overview = data_col.find('div', class_='overview')
        if overview:
            txt_div = overview.find('div', class_='dataTxt')
            if txt_div:
                html_effect = txt_div.decode_contents().strip()
                data['effect'] = html_effect

    return data


# ==========================================
# Extraction Loop for GD01
# ==========================================

target_package_id = '616101' # GD01
search_url = "https://www.gundam-gcg.com/en/cards/index.php?search=true"
headers = {'User-Agent': 'Mozilla/5.0'}

print(f"Fetching card list for GD01 (Package ID: {target_package_id})...")
# Make a POST request to search for the specific set cards
response = requests.post(search_url, headers=headers, data={'package': target_package_id})
soup = BeautifulSoup(response.content, 'html.parser')

# Extract card links from the search results
card_links = soup.find_all('a', class_='cardStr')
card_ids = []
for link in card_links:
    data_src = link.get('data-src', '')
    if 'detailSearch=' in data_src:
        parsed = urllib.parse.urlparse(data_src)
        qs = urllib.parse.parse_qs(parsed.query)
        if 'detailSearch' in qs:
            card_ids.append(qs['detailSearch'][0])

# Remove duplicates maintaining order
card_ids = list(dict.fromkeys(card_ids))
total_cards = len(card_ids)
print(f"Found {total_cards} unique cards in GD01. Starting extraction...")

all_cards_data = []

# Iterate through all cards
for i, cid in enumerate(card_ids, 1):
    print(f"[{i}/{total_cards}] Extracting data for {cid}...")
    try:
        card_info = get_card_details(cid)
        all_cards_data.append(card_info)
    except Exception as e:
        print(f"[{i}/{total_cards}] Error extracting {cid}: {e}")
        
    # Be mindful of the server: add a sleep delay (1.5 seconds) between requests!
    time.sleep(1.5)
    
# Save our collected JSON array to a file
output_filename = "gd01_cards.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(all_cards_data, f, indent=2, ensure_ascii=False)
    
print(f"\nExtraction complete! Saved {len(all_cards_data)} cards to '{output_filename}'.")

In [ ]:
import os
import re
import json
import time
import requests
import urllib.parse
from bs4 import BeautifulSoup

# Relying on `sets` from Cell 1 and `get_card_details` from Cell 2
search_url = "https://www.gundam-gcg.com/en/cards/index.php?search=true"
headers = {'User-Agent': 'Mozilla/5.0'}

# Create a folder to keep the JSON files organized
output_dir = "sets_data"
os.makedirs(output_dir, exist_ok=True)

print(f"Ready to iterate through {len(sets)} sets.")
print("The script will pause before each set. Follow the prompts in the input box.\n")

for set_val, set_label in sets.items():
    # Use the set's bracket ID (e.g., "[GD01]") or a shortcut name for special sets
    if '[' in set_label and ']' in set_label:
        clean_name = set_label.split('[')[1].replace(']', '').strip().lower()
    else:
        lower_label = set_label.lower()
        if 'other' in lower_label: clean_name = 'other'
        elif 'beta' in lower_label: clean_name = 'beta'
        elif 'basic' in lower_label: clean_name = 'basic'
        elif 'promotion' in lower_label or 'promo' in lower_label: clean_name = 'promo'
        else:
            clean_name = re.sub(r'[^a-zA-Z0-9]', '_', set_label).strip('_').lower()
            clean_name = re.sub(r'_+', '_', clean_name)
            
    output_file = os.path.join(output_dir, f"{clean_name}.json")
    
    # Skip if we already scraped this successfully!
    if os.path.exists(output_file):
        print(f"--- Already have {output_file} for '{set_label}', skipping! ---")
        continue
    
    # -------------------------------------------------------------
    # Interactive Prompt: Waits for the user before proceeding
    # -------------------------------------------------------------
    prompt_msg = f"Next Set: {set_label}\nPress Enter to scrape, type 'skip' to bypass, or 'q' to quit:"
    ans = input(prompt_msg).strip().lower()
    
    if ans == 'q':
        print("Scraping stopped by user.")
        break
    elif ans == 'skip':
        print(f"Skipping {set_label}...\n")
        continue

    print(f"\nFetching list for {set_label}...")
    response = requests.post(search_url, headers=headers, data={'package': set_val})
    soup = BeautifulSoup(response.content, 'html.parser')

    card_links = soup.find_all('a', class_='cardStr')
    card_ids = []
    for link in card_links:
        ds = link.get('data-src', '')
        if 'detailSearch=' in ds:
            qs = urllib.parse.parse_qs(urllib.parse.urlparse(ds).query)
            if 'detailSearch' in qs:
                card_ids.append(qs['detailSearch'][0])

    card_ids = list(dict.fromkeys(card_ids))
    total_cards = len(card_ids)
    
    if total_cards == 0:
        print(f"No cards found for {set_label}.\n")
        continue
        
    print(f"Found {total_cards} cards. Starting extraction...")
    
    set_data = []
    for i, cid in enumerate(card_ids, 1):
        print(f"  [{i}/{total_cards}] Extracting {cid}...")
        try:
            card_info = get_card_details(cid)
            set_data.append(card_info)
        except Exception as e:
            print(f"  --> Error extracting {cid}: {e}")
            
        time.sleep(1.5)  # Be kind to the server!
        
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(set_data, f, indent=2, ensure_ascii=False)
        
    print(f"Done! Saved {total_cards} cards to '{output_file}'\n")